# PaySim — Exploratory Data Analysis

**Playbook:** `6_eda_playbook.docx` · **Batch 1 — Phases 1–4**  
**Author:** Sofia Hansari · **Date:** 2026-10-20  
**Next notebook:** Modelling — do not train models here.

> All analysis uses only information available at prediction time.  
> No model training, SHAP, precision/recall, or threshold selection in this notebook.

---
**Phases covered in this batch:**
- Phase 1 — Problem Definition and Target Establishment (Sections 2–4)
- Phase 2 — Data Loading and Initial Profiling (Sections 5–7)
- Phase 3 — Sweetviz Reports (Sections 8–10)
- Phase 4 — Data Quality Assessment (Sections 11–15)

**Batch 2** covers Phases 5–9 (univariate, bivariate, time-series, drift, correlation).  
**Batch 3** covers Phases 10–13 (RFM candidates, feature register, validation, handover).

### ---
## Phase 1 — Problem Definition and Target Establishment

**Playbook sections 2–4.**  
Purpose: document exactly what we are predicting, verify the target column, and establish the fraud rate before any feature analysis.  
**Gate:** do not run target-dependent analysis until the target has been verified (end of this phase).

In [3]:
# Section 2 — Business problem documentation
# Verify each entry matches the actual data before proceeding.

PROBLEM_DEFINITION = {
    'business_objective'       : 'Detect fraudulent mobile money transactions before they complete',
    'unit_of_prediction'       : 'Individual transaction (one row = one transaction)',
    'prediction_time_boundary' : 'Features available when the transaction is submitted, before settlement',
    'intended_use'             : 'Flag high-risk transactions for manual review or automatic blocking',
    'batch_or_individual'      : 'Batch during training; individual at inference time',
    'expected_outcome'         : 'Binary: isFraud = 1 (fraudulent) or 0 (legitimate)',
    'dataset_source'           : 'PaySim synthetic mobile-money simulator (calibrated on M-Pesa data, Kaggle)',
    'known_limitations'        : [
        'Synthetic data — patterns may not fully reflect real-world fraud',
        'isFlaggedFraud is a simulation artefact, not a real detection system output',
        'step represents simulated hours (1 step = 1 hour), not real calendar time',
        'Only TRANSFER and CASH_OUT types contain fraud — simulation design choice',
    ],
}

print('BUSINESS PROBLEM DEFINITION')
print('=' * 65)
for k, v in PROBLEM_DEFINITION.items():
    if isinstance(v, list):
        print(f'\n{k}:')
        for item in v:
            print(f'  • {item}')
    else:
        print(f'{k:32s}: {v}')

BUSINESS PROBLEM DEFINITION
business_objective              : Detect fraudulent mobile money transactions before they complete
unit_of_prediction              : Individual transaction (one row = one transaction)
prediction_time_boundary        : Features available when the transaction is submitted, before settlement
intended_use                    : Flag high-risk transactions for manual review or automatic blocking
batch_or_individual             : Batch during training; individual at inference time
expected_outcome                : Binary: isFraud = 1 (fraudulent) or 0 (legitimate)
dataset_source                  : PaySim synthetic mobile-money simulator (calibrated on M-Pesa data, Kaggle)

known_limitations:
  • Synthetic data — patterns may not fully reflect real-world fraud
  • isFlaggedFraud is a simulation artefact, not a real detection system output
  • step represents simulated hours (1 step = 1 hour), not real calendar time
  • Only TRANSFER and CASH_OUT types contain fraud —

### Section 3 — Prediction Task

| Field | Value |
|---|---|
| **Task type** | Binary classification |
| **Target column** | `isFraud` |
| **Class 1** | Confirmed fraudulent transaction (PaySim simulation label) |
| **Class 0** | Confirmed legitimate transaction |
| **Label source** | PaySim simulation engine — ground truth by construction |
| **Label available at prediction time?** | **No** — this is what we are predicting |
| **Valid target values** | 0, 1 only |
| **Target derivation** | Existing and valid — no derivation required |
| **Caution** | Do not assume an uninvestigated transaction is legitimate |

**Target validation register** (complete after running the cell below):

| Field | Value |
|---|---|
| Target definition | `isFraud` = 1 means the transaction was fraudulent |
| Total records | *(fill in after loading)* |
| Valid labelled records | *(fill in)* |
| Class 1 count and % | *(fill in)* |
| Class 0 count and % | *(fill in)* |
| Missing / invalid target | *(fill in)* |
| Unresolved issues | *(fill in)* |

---
## Phase 2 — Data Loading and Initial Profiling

**Playbook sections 5–7.**

In [4]:
# Section 5 — Notebook setup
import sys
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Robust project root finder — searches upward for pyproject.toml
def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists():
            return p
    raise FileNotFoundError('Could not find project root. Is pyproject.toml present?')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from fraud_detection.common.paths import PAYSIM_CSV, reports_dir

REPORTS_DIR = reports_dir('data_quality')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 50)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print(f'Project root : {PROJECT_ROOT}')
print(f'PaySim CSV   : {PAYSIM_CSV}')
print(f'File exists  : {PAYSIM_CSV.exists()}')
print(f'Reports dir  : {REPORTS_DIR}')

# Helper: feature binding rate
def binding_rate(series: pd.Series) -> float:
    """% of rows where the feature is non-null and non-zero."""
    return ((series.notna()) & (series != 0)).mean() * 100

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# Section 6 — Load and inspect the dataset
# PaySim has ~6.3M rows. Loading takes ~10 seconds and uses ~1.5 GB RAM.
# We preserve the raw DataFrame (df_raw) and work on a copy (df).

print('Loading PaySim CSV...')
df_raw = pd.read_csv(PAYSIM_CSV)
df = df_raw.copy()

print(f'Shape       : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
df.head(5)

In [ ]:
# Schema: column names, data types, step range
print('COLUMN NAMES AND DATA TYPES')
print('=' * 45)
print(df.dtypes)
print(f'\nStep range  : {df["step"].min()} \u2192 {df["step"].max()}')
print(f'Unique steps: {df["step"].nunique()}')

In [ ]:
# Missing values, unique counts, duplicate rows
print('MISSING VALUES')
print('-' * 40)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(4)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
has_missing = missing_df[missing_df['missing_count'] > 0]
print(has_missing if len(has_missing) > 0 else 'None — no missing values found.')

print('\nUNIQUE VALUE COUNTS PER COLUMN')
print('-' * 40)
print(df.nunique().rename('unique_count').to_string())

print('\nDUPLICATE ROWS')
print('-' * 40)
dup_count = df.duplicated().sum()
print(f'Exact duplicate rows: {dup_count:,}')

In [ ]:
# Section 4 gate: validate target before any target-dependent analysis
print('TARGET VALIDATION GATE')
print('=' * 50)

target_col = 'isFraud'
valid_values = {0, 1}
invalid_mask = ~df[target_col].isin(valid_values)
missing_target = df[target_col].isnull().sum()

print(f'Total rows              : {len(df):,}')
print(f'Missing target values   : {missing_target:,}')
print(f'Invalid target values   : {invalid_mask.sum():,}')
print(f'Valid labelled records  : {(~invalid_mask & ~df[target_col].isnull()).sum():,}')

fraud_counts = df[target_col].value_counts().sort_index()
fraud_pct    = df[target_col].value_counts(normalize=True).sort_index() * 100
target_summary = pd.DataFrame({'count': fraud_counts, 'pct': fraud_pct.round(4)})
print('\nClass distribution:')
print(target_summary.rename(index={0: 'Class 0 (Legitimate)', 1: 'Class 1 (Fraud)'}))

n_neg = fraud_counts[0]
n_pos = fraud_counts[1]
spw   = n_neg / n_pos
print(f'\nClass ratio (neg:pos)   : {n_neg:,} : {n_pos:,}')
print(f'scale_pos_weight (full) : {spw:.1f}  <-- placeholder; recalculate on training split in Phase 2')

if missing_target == 0 and invalid_mask.sum() == 0:
    print('\n[GATE PASSED] Target is valid. Proceeding with target-dependent analysis.')
else:
    print('\n[GATE FAILED] Fix target issues before continuing.')

In [ ]:
# Numerical summary statistics
print('NUMERICAL SUMMARY STATISTICS')
df.describe().T

In [ ]:
# Categorical feature summaries
print('CATEGORICAL FEATURE SUMMARIES')
print('=' * 50)
for col in df.select_dtypes(include='object').columns:
    print(f'\n{col}  ({df[col].nunique():,} unique values):')
    vc = df[col].value_counts()
    pct = df[col].value_counts(normalize=True) * 100
    summary = pd.DataFrame({'count': vc, 'pct': pct.round(2)}).head(10)
    print(summary.to_string())

### Section 7 — Data Dictionary

| Feature | Type | Meaning | Unit | Role | Available at prediction time? | Notes |
|---|---|---|---|---|---|---|
| `step` | int | Simulated time unit (1 step ≈ 1 hour) | Simulated hours | Time | Yes | Not a real calendar timestamp; range 1–743 |
| `type` | str | Transaction type | Category | Feature | Yes | CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER |
| `amount` | float | Transaction amount | Currency (M-Pesa) | Feature | Yes | Right-skewed; log-transform likely needed |
| `nameOrig` | str | Sender account ID | Identifier | Graph construction only | Yes (account exists) | High cardinality; not a predictive feature |
| `oldbalanceOrg` | float | Sender balance **before** transaction | Currency | Feature (investigate) | Yes — pre-transaction | Zero-heavy; investigate leakage |
| `newbalanceOrig` | float | Sender balance **after** transaction | Currency | **Exclude (leakage risk)** | **No** — post-settlement | Only known after transaction completes |
| `nameDest` | str | Recipient account ID | Identifier | Graph construction only | Yes (account exists) | High cardinality; not a predictive feature |
| `oldbalanceDest` | float | Recipient balance **before** transaction | Currency | Feature (investigate) | Investigate | Zero-heavy for merchants |
| `newbalanceDest` | float | Recipient balance **after** transaction | Currency | **Exclude (leakage risk)** | **No** — post-settlement | Only known after transaction completes |
| `isFraud` | int | Ground-truth fraud label | Binary 0/1 | Target | **No** — this is what we predict | |
| `isFlaggedFraud` | int | Simulation rule-based flag (amount > 200k + balance drained) | Binary 0/1 | **Investigate (leakage risk)** | **No** — simulation artefact | Investigate overlap with `isFraud` |

**Leakage risk summary:**  
- `newbalanceOrig` and `newbalanceDest` — post-settlement balances, never use as features  
- `isFlaggedFraud` — simulation rule applied after fraud is known; investigate before using  
- `nameOrig` / `nameDest` — identifiers only, used for graph construction

---
## Phase 3 — Sweetviz Reports

**Playbook sections 8–10.**  
Two HTML reports are generated:
1. `sweetviz_overall_report.html` — full dataset overview  
2. `sweetviz_class1_vs_class0_report.html` — Fraud vs Legitimate comparison

> **Sampling:** Sweetviz on 6.3M rows is very slow. We use a **stratified sample of 100,000 rows** (preserving the fraud rate) for both reports.  
> `nameOrig` and `nameDest` are excluded from the reports — they are extremely high-cardinality identifiers that add no analytical value in Sweetviz.

After the cells run, **open both HTML files in your browser** and fill in the findings register (Section 10).

In [ ]:
# Build stratified sample for Sweetviz
SAMPLE_SIZE = 100_000

fraud_rows = df[df['isFraud'] == 1]
legit_rows = df[df['isFraud'] == 0]

n_fraud = min(len(fraud_rows), int(SAMPLE_SIZE * len(fraud_rows) / len(df)))
n_legit = SAMPLE_SIZE - n_fraud

sv_sample = pd.concat([
    fraud_rows.sample(n_fraud, random_state=42),
    legit_rows.sample(n_legit, random_state=42)
]).reset_index(drop=True)

# Exclude identifier columns from Sweetviz
sv_cols = [c for c in sv_sample.columns if c not in ['nameOrig', 'nameDest']]
sv_data = sv_sample[sv_cols]

print(f'Full dataset — rows: {len(df):,} | fraud rate: {df["isFraud"].mean()*100:.4f}%')
print(f'Sample       — rows: {len(sv_sample):,} | fraud rate: {sv_sample["isFraud"].mean()*100:.4f}%')
print(f'  Fraud in sample : {n_fraud:,}')
print(f'  Legit in sample : {n_legit:,}')
print(f'Columns in report: {sv_cols}')

In [ ]:
# Section 8 — Sweetviz Report 1: Overall dataset
import sweetviz as sv

overall_path = str(REPORTS_DIR / 'sweetviz_overall_report.html')
print('Generating overall Sweetviz report...')
report_overall = sv.analyze(sv_data, target_feat='isFraud')
report_overall.show_html(overall_path, open_browser=False)
print(f'Saved: {overall_path}')
print('Open this file in your browser to review distributions, missingness, and target associations.')

In [ ]:
# Section 9 — Sweetviz Report 2: Class 1 vs Class 0
fraud_sv  = sv_data[sv_data['isFraud'] == 1]
legit_sv  = sv_data[sv_data['isFraud'] == 0]

print(f'Fraud sample : {len(fraud_sv):,} rows')
print(f'Legit sample : {len(legit_sv):,} rows')

compare_path = str(REPORTS_DIR / 'sweetviz_class1_vs_class0_report.html')
print('Generating comparison Sweetviz report...')
report_compare = sv.compare(
    [fraud_sv, 'Fraud (Class 1)'],
    [legit_sv, 'Legitimate (Class 0)']
)
report_compare.show_html(compare_path, open_browser=False)
print(f'Saved: {compare_path}')
print('Open this file in your browser to compare fraud vs legitimate distributions.')

### Section 10 — Sweetviz Findings Register

**Fill in after reviewing both HTML reports.**  
Open `reports/data_quality/sweetviz_overall_report.html` and `reports/data_quality/sweetviz_class1_vs_class0_report.html` in your browser.

Pay special attention to: `amount`, `oldbalanceOrg`, `oldbalanceDest`, `type`, `isFlaggedFraud`.

| # | Feature | Observed difference | Interpretation | Leakage concern | Follow-up | Status |
|---|---|---|---|---|---|---|
| 1 | `amount` | | | | | Open |
| 2 | `oldbalanceOrg` | | | | | Open |
| 3 | `oldbalanceDest` | | | | | Open |
| 4 | `type` | | | | | Open |
| 5 | `isFlaggedFraud` | | | | | Open |
| 6 | | | | | | Open |

> A visible difference between classes does not establish predictive usefulness or causation. Record observations only; interpretation comes in Phase 6 (bivariate analysis).

---
## Phase 4 — Data Quality Assessment

**Playbook sections 11–15.**  
Covers: missingness, feature binding rates, duplicates, invalid values, consistency checks, outlier analysis.  
**Output:** cleaning recommendation register (Section 15) at the end of this phase.

In [ ]:
# Section 11 — Missingness and zero-value analysis by class
# For PaySim: no nulls exist, but zero values are a form of missingness for balance columns.

balance_cols = ['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
numeric_cols = ['amount'] + balance_cols

print('ZERO VALUES (%) BY FRAUD LABEL')
print('A zero balance may indicate: merchant account, new account, or fraudster zeroing out.')
print('-' * 65)

rows = []
for col in balance_cols:
    pct_0_legit = (df[df['isFraud']==0][col] == 0).mean() * 100
    pct_0_fraud = (df[df['isFraud']==1][col] == 0).mean() * 100
    rows.append({'column': col, 'Legit zero %': round(pct_0_legit, 2), 'Fraud zero %': round(pct_0_fraud, 2)})

zero_df = pd.DataFrame(rows).set_index('column')
zero_df['Difference'] = (zero_df['Fraud zero %'] - zero_df['Legit zero %']).round(2)
print(zero_df.to_string())

In [ ]:
# Section 11 continued — Feature binding rates
# Binding rate = % of rows with a usable (non-null AND non-zero) value.
# A non-null value may still be unusable (e.g. zero balance encodes 'no balance data').

print('FEATURE BINDING RATES (% rows with non-null, non-zero value)')
print('-' * 65)

br_overall = {col: binding_rate(df[col]) for col in numeric_cols}
br_fraud   = {col: binding_rate(df[df['isFraud']==1][col]) for col in numeric_cols}
br_legit   = {col: binding_rate(df[df['isFraud']==0][col]) for col in numeric_cols}

br_df = pd.DataFrame({
    'Overall (%)' : br_overall,
    'Legit (%)  ' : br_legit,
    'Fraud (%)  ' : br_fraud,
}).round(2)
print(br_df.to_string())

print('\nInterpretation:')
print('  amount           : always > 0 (every transaction has an amount)')
print('  oldbalanceOrg    : sender opening balance; may be 0 for new/empty accounts')
print('  newbalanceOrig   : sender closing balance; 0 could mean account was drained (fraud signal)')
print('  oldbalanceDest   : recipient opening balance; 0 common for merchant accounts')
print('  newbalanceDest   : recipient closing balance; 0 suspicious on recipients')

In [ ]:
# Section 29 preview — Binding rate trends over time (included here per playbook Section 11)
# Abrupt changes in binding rate suggest data pipeline or schema changes.

step_br = pd.DataFrame({
    col: df.groupby('step')[col].apply(binding_rate)
    for col in numeric_cols
})
rolling_br = step_br.rolling(10, center=True).mean()

fig, ax = plt.subplots(figsize=(13, 4))
for col in numeric_cols:
    ax.plot(rolling_br.index, rolling_br[col], label=col, lw=1.4)
ax.set_xlabel('Step (simulated hour)')
ax.set_ylabel('Binding rate (%)')
ax.set_title('Feature Binding Rates Over Time (10-step rolling average)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print('Flag: any column whose binding rate drops or spikes abruptly has a data quality issue.')

In [ ]:
# Section 12 — Duplicate analysis
print('DUPLICATE ANALYSIS')
print('=' * 55)

# Exact row duplicates
exact_dups = df.duplicated().sum()
print(f'Exact duplicate rows                : {exact_dups:,}')

# Duplicate transaction signatures (same step + type + amount + nameOrig + nameDest)
sig_cols = ['step', 'type', 'amount', 'nameOrig', 'nameDest']
sig_dups = df.duplicated(subset=sig_cols).sum()
print(f'Duplicate transaction signatures    : {sig_dups:,}')
print(f'  (same step + type + amount + nameOrig + nameDest)')

# Check for conflicting labels on same signature
if sig_dups > 0:
    dup_mask = df.duplicated(subset=sig_cols, keep=False)
    label_conflicts = df[dup_mask].groupby(sig_cols)['isFraud'].nunique()
    n_conflicts = (label_conflicts > 1).sum()
    print(f'Signatures with conflicting labels  : {n_conflicts:,}')

# Repeated origin-destination pairs (same sender + recipient across different steps)
pair_counts = df.groupby(['nameOrig', 'nameDest']).size()
repeated_pairs = (pair_counts > 1).sum()
print(f'\nRepeated origin-destination pairs   : {repeated_pairs:,}')
print(f'Max transactions for one O-D pair   : {pair_counts.max():,}')
print('\nNote: repeated pairs are expected and potentially informative (RFM feature candidate).')

In [ ]:
# Section 13 — Invalid values and consistency checks
print('INVALID VALUES AND CONSISTENCY CHECKS')
print('=' * 60)

# 1. Transaction types
valid_types = {'CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER'}
invalid_types = ~df['type'].isin(valid_types)
print(f'1. Invalid transaction types : {invalid_types.sum():,}')

# 2. Negative amounts
neg_amount = (df['amount'] < 0).sum()
print(f'2. Negative amounts          : {neg_amount:,}')

# 3. Negative balances
for col in balance_cols:
    neg_bal = (df[col] < 0).sum()
    print(f'3. Negative {col:18s}: {neg_bal:,}')

# 4. isFraud outside {0,1}
invalid_label = ~df['isFraud'].isin([0, 1])
print(f'4. Invalid isFraud values    : {invalid_label.sum():,}')

# 5. isFlaggedFraud outside {0,1}
invalid_flag = ~df['isFlaggedFraud'].isin([0, 1])
print(f'5. Invalid isFlaggedFraud    : {invalid_flag.sum():,}')

# 6. isFlaggedFraud vs isFraud relationship (leakage investigation)
print('\nisFlaggedFraud vs isFraud crosstab:')
ct = pd.crosstab(df['isFlaggedFraud'], df['isFraud'],
                 rownames=['isFlaggedFraud'], colnames=['isFraud'], margins=True)
print(ct)

flagged = df[df['isFlaggedFraud'] == 1]
print(f'\nOf {len(flagged):,} flagged rows:')
print(f'  {flagged["isFraud"].sum():,} are actual fraud ({flagged["isFraud"].mean()*100:.1f}%)')
print(f'  {(1 - flagged["isFraud"]).sum():.0f} are NOT fraud (false positives from the simulation rule)')
print('\nConclusion candidate: isFlaggedFraud is a noisy proxy for isFraud. EXCLUDE from all models.')

# 7. Balance conservation check (TRANSFER + CASH_OUT only)
print('\nBalance conservation check (TRANSFER + CASH_OUT):')
tf_co = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
tf_co['orig_error'] = (tf_co['oldbalanceOrg'] - tf_co['newbalanceOrig'] - tf_co['amount']).abs()
tf_co['dest_error'] = (tf_co['newbalanceDest'] - tf_co['oldbalanceDest'] - tf_co['amount']).abs()

tol = 0.01
orig_viol = (tf_co['orig_error'] > tol).sum()
dest_viol = (tf_co['dest_error'] > tol).sum()
print(f'  Sender side violations   : {orig_viol:,} / {len(tf_co):,} ({orig_viol/len(tf_co)*100:.1f}%)')
print(f'  Recipient side violations: {dest_viol:,} / {len(tf_co):,} ({dest_viol/len(tf_co)*100:.1f}%)')

# Break down violations by fraud label
print('\nSender-side violations by fraud label:')
print(tf_co.groupby('isFraud')['orig_error'].apply(lambda x: (x > tol).mean() * 100).round(2).rename('violation_rate_pct'))

In [ ]:
# Section 14 — Outlier analysis
# Histograms, ECDF plots, log-scale distributions, box plots.
# Do NOT remove extreme transactions — they may represent important fraud behaviour.

fig, axes = plt.subplots(3, 2, figsize=(14, 14))

# Row 1: amount raw (capped) and log1p
for label, color, lname in [(0, 'steelblue', 'Legitimate'), (1, 'tomato', 'Fraud')]:
    s = df[df['isFraud'] == label]['amount']
    axes[0, 0].hist(s.clip(upper=s.quantile(0.99)), bins=80, alpha=0.55,
                    color=color, label=lname, density=True)
    axes[0, 1].hist(np.log1p(s), bins=80, alpha=0.55,
                    color=color, label=lname, density=True)

axes[0, 0].set_title('amount (raw, capped at 99th pct)')
axes[0, 1].set_title('amount (log1p transformed)')
for ax in axes[0]:
    ax.legend(); ax.set_xlabel('Amount')

# Row 2: ECDF for amount and log1p(amount)
for label, color, lname in [(0, 'steelblue', 'Legitimate'), (1, 'tomato', 'Fraud')]:
    s = df[df['isFraud'] == label]['amount']
    s_samp = s.sample(min(len(s), 10_000), random_state=42)
    s_sorted = np.sort(s_samp)
    axes[1, 0].plot(s_sorted, np.linspace(0, 1, len(s_sorted)), color=color, label=lname, lw=1.5)
    axes[1, 1].plot(np.log1p(s_sorted), np.linspace(0, 1, len(s_sorted)), color=color, label=lname, lw=1.5)

axes[1, 0].set_title('ECDF — amount (raw)')
axes[1, 1].set_title('ECDF — log1p(amount)')
for ax in axes[1]:
    ax.legend(); ax.set_xlabel('Amount'); ax.set_ylabel('Cumulative probability')

# Row 3: log1p(balance) box plots + amount quantile comparison
tf_plot = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])][['isFraud', 'oldbalanceOrg', 'oldbalanceDest']].copy()
tf_melt = tf_plot.melt(id_vars='isFraud', var_name='balance_col', value_name='value')
tf_melt['log1p_value'] = np.log1p(tf_melt['value'])
tf_melt['Class'] = tf_melt['isFraud'].map({0: 'Legitimate', 1: 'Fraud'})

sns.boxplot(data=tf_melt, x='balance_col', y='log1p_value', hue='Class',
            palette={'Legitimate': 'steelblue', 'Fraud': 'tomato'}, ax=axes[2, 0])
axes[2, 0].set_title('log1p(opening balance) by class (TRANSFER+CASH_OUT)')
axes[2, 0].set_xlabel('')

qtls = [0.5, 0.75, 0.9, 0.95, 0.99, 0.999]
q_fraud = np.log1p(df[df['isFraud']==1]['amount'].quantile(qtls).values)
q_legit = np.log1p(df[df['isFraud']==0]['amount'].quantile(qtls).values)
x = np.arange(len(qtls))
axes[2, 1].bar(x - 0.2, q_legit, width=0.35, color='steelblue', label='Legitimate', alpha=0.8)
axes[2, 1].bar(x + 0.2, q_fraud, width=0.35, color='tomato',    label='Fraud',       alpha=0.8)
axes[2, 1].set_xticks(x)
axes[2, 1].set_xticklabels([str(q) for q in qtls])
axes[2, 1].set_title('log1p(amount) quantile comparison by class')
axes[2, 1].set_xlabel('Quantile'); axes[2, 1].legend()

plt.suptitle('Outlier Analysis — Amount and Opening Balances', y=1.01, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key questions from this chart:')
print('  1. Do fraud transactions have a different amount distribution than legitimate ones?')
print('  2. Is the log1p transform sufficient to normalise amount, or is further handling needed?')
print('  3. Do fraud transactions show near-zero opening balances (account draining pattern)?')

### Section 15 — Cleaning Recommendation Register

Update this table based on what you observed above. Add rows for new issues.

| # | Affected field(s) | Evidence | Severity | Proposed action | When |
|---|---|---|---|---|---|
| 1 | `newbalanceOrig`, `newbalanceDest` | Post-settlement values — not available at prediction time | **High** | Exclude from all models | Phase 2 preprocessing |
| 2 | `isFlaggedFraud` | Simulation artefact; near-perfect overlap with `isFraud` | **High** | Exclude from all models | Phase 2 preprocessing |
| 3 | `nameOrig`, `nameDest` | High-cardinality identifiers — not predictive features | Medium | Exclude as features; use for graph construction only | Phase 2 preprocessing |
| 4 | `amount` | Highly right-skewed | Low | Apply log1p transform during preprocessing | Phase 2 preprocessing |
| 5 | `oldbalanceDest` | Zero-heavy (merchants) | Low | Investigate further in Phase 6 (bivariate); possibly add binary `dest_has_history` flag | Phase 3 feature engineering |
| 6 | Balance conservation violations | Sender-side violations concentrate in fraud rows | Medium | Derive `balance_discrepancy` as a feature candidate | Phase 3 feature engineering |

> **Rule:** Keep the EDA notebook focused on diagnosis and recommendations.  
> Do not silently overwrite the raw data here. All cleaning happens in Phase 2 (preprocessing pipeline).

---
## GPT/Claude Review Checkpoint 1 — Dataset Understanding

**Playbook instruction:** Provide a compact summary; ask Claude to identify missing checks, challenge interpretations, and flag leakage risks.

**Summary of findings so far** *(fill in after running all cells above)*:

| Item | Finding |
|---|---|
| Dataset shape | *(rows x cols)* |
| Missing values | *(count)* |
| Duplicate rows | *(count)* |
| Fraud rate | *(%)*  |
| scale_pos_weight | *(placeholder; finalise on training split)* |
| Transaction types with fraud | TRANSFER, CASH_OUT |
| Step range | 1 → 743 (simulated hours, ~31 days) |
| Balance conservation violations | *(% in fraud vs legitimate)* |
| Leakage risks identified | `newbalanceOrig`, `newbalanceDest`, `isFlaggedFraud` |
| Sweetviz reports | *(open in browser and fill in Section 10 register)* |

**Questions to bring to Claude review:**
1. Do balance conservation violations concentrate in fraud rows? What does that imply for feature engineering?
2. Is `oldbalanceOrg` (pre-transaction sender balance) safe to use, or does it also encode fraud signal via draining?
3. Are zero `oldbalanceDest` values a structural merchant-account feature or a data quality issue?
4. Should `step` be used as a raw feature, cyclically encoded, or only for splitting?

---
**Batch 1 complete.**  
Run all cells top-to-bottom, open both Sweetviz HTML reports, fill in the registers in Sections 10 and 15, then say **“Batch 2 ready”** to continue with Phases 5–9 (univariate, bivariate, time-series, drift, correlation).